In [0]:
# %run /Workspace/Repos/projects/databricks-batch-pipeline/modules/silver_layer

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
from src.pipeline_core.framework.silver.silver_layer import SilverLayer

In [0]:
dbutils.widgets.text("pipeline_name", "")
pipeline_name = dbutils.widgets.get("pipeline_name")

In [0]:
silver_pipeline = SilverLayer.from_config_table(spark, pipeline_name)

In [0]:
# data = {
#     "pipeline_name": "books",
#     "table_name": "books",
#     "schema_details": {
#         "title": "string",
#         "subtitle": "string",
#         "authors": "string",
#         "categories": "string",
#         "published_year": "int",
#         "average_rating": "double",
#         "num_pages": "int",
#         "ratings_count": "int"
#     },
#     "keys": ["title"],
#     "write_mode": "scd1"        
# }

In [0]:
# data = {
#     "table_name": "books2",
#     "schema_details": {
#         "title": "string",
#         "subtitle": "string",
#         "authors": "string",
#         "categories": "string",
#         "published_year": "int",
#         "average_rating": "double",
#         "num_pages": "int",
#         "ratings_count": "int"
#     },
#     "keys": ["title"],
#     "write_mode": "scd2"        
# }

In [0]:
# silver_pipeline = SilverLayer(
#     pipeline_name = data["pipeline_name"],
#     table_name = data["table_name"],
#     schema_details = data["schema_details"],
#     keys = data["keys"],
#     write_mode = data["write_mode"]
# )

df = silver_pipeline.read_data()
invalid_df = silver_pipeline.get_invalid_record(df)
key_null_df = silver_pipeline.get_key_null_record(df)
dup_df = silver_pipeline.get_dup_record(df, key_null_df)
all_bad_df = silver_pipeline.get_all_bad_records(
    invalid_df = invalid_df, 
    key_null_df = key_null_df, 
    duplicate_df = dup_df
)
cleaned_df = silver_pipeline.get_cleansing_df(df, all_bad_df)
load_bad_records = silver_pipeline.load_bad_record(all_bad_df)
load_to_silver_layer = silver_pipeline.load_to_silver_layer(cleaned_df)




In [0]:
cleaned_df.display()

In [0]:

(
    spark.table("hr_silver")
            # .filter(col('end_date').isNotNull())
            # .filter(col("Employee_name") == "Wilson  K")
            # .isNotNull())
            # .orderBy(desc(col("last_mod_ts")))
            .display()
)


In [0]:
%sql
select * from hr_bronze

In [0]:
%sql
select * from workspace.default.hr_bad_record